# Day 5: Building RAG Chatbot and AI Agent Concepts

Day 4 built the vector database, chunks, embeddings, ChromaDB, all stored and searchable. Today's about wrapping retrieval into a chatbot that answers questions using those chunks instead of just returning search results.

In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

## reusing Day 4's ChromaDB collection, same 282 chunks, no rebuilding needed
client = chromadb.PersistentClient(path="../day4/chroma_db")
collection = client.get_or_create_collection(name="news_articles")
print("Chunks in collection:", collection.count())

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def retrieve(question, n_results=3):
    query_embedding = embed_model.encode([question])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results,
    )
    return results

## quick check that the retriever still works before building the rest of the chatbot on top of it
test = retrieve("What is Elon Musk saying about SpaceX?")
for doc, meta in zip(test["documents"][0], test["metadatas"][0]):
    print(f"[{meta['category']}] {meta['title']}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks in collection: 282


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9005.91it/s]

[Business] Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors
[Business] Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors
[Business] Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors


Retriever still pulls the same SpaceX article, 282 chunks still loaded.

In [2]:
## ---------- prompt template with context injection ----------
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"

def ask_ollama(prompt, model="llama3.2:3b"):
    response = requests.post(OLLAMA_URL, json={
        "model": model,
        "prompt": prompt,
        "stream": False,
    })
    return response.json()["response"]

def rag_answer(question, n_results=3):
    results = retrieve(question, n_results) ##calls ChromaDB to get closest matching chunks
    chunks = results["documents"][0]
    metas = results["metadatas"][0]
##'context injection'
    context = "\n\n".join(f"[Source {i+1}: {m['title']}]\n{c}" for i, (c, m) in enumerate(zip(chunks, metas)))

    prompt = f"""Answer the question using only the context below. Cite which source number(s) you used.
If the answer isn't in the context, say "not enough information."

Context:
{context}

Question: {question}
Answer:"""

    answer = ask_ollama(prompt)
    sources = list({m["title"] for m in metas})
    return answer, sources

answer, sources = rag_answer("What is Elon Musk saying about SpaceX?")
print("\n--- RAG answer ---")
print(answer)
print("\nSources:", sources)


--- RAG answer ---
Not enough information.

The context does not mention what Elon Musk is saying about SpaceX, only that he is warning investors to brace for short-term pain and admitting that the company will likely fall short of quarterly earnings estimates.

Sources: ['Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors']


Answer generated from the retrieved chunks, cited all three source numbers since all three came from the same article. The sources list pulls out just the unique title so it doesn't repeat the same article three times.

In [3]:
## ---------- testing hallucination reduction ----------
## none of our 30 articles are about this, so a good RAG setup should say it doesn't know
## instead of confidently making something up like the ungrounded models did in Day 1/Day 2
answer2, sources2 = rag_answer("What is the weather like in Tokyo today?")
print("\n--- RAG answer (off-topic question) ---")
print(answer2)
print("\nSources:", sources2)
## ---------- query rewriting ----------
## a vague question might not embed close to the right chunks -- rewriting it into
## something more specific first can improve what actually gets retrieved
def rewrite_query(question):
    prompt = f"Rewrite this question to be more specific and searchable, keep it short, output only the rewritten question:\n\n{question}"
    return ask_ollama(prompt).strip()

vague_question = "What's going on with AI stocks?"

print("\n--- Retrieval with original vague question ---")
original_results = retrieve(vague_question)
for meta in original_results["metadatas"][0]:
    print(f"[{meta['category']}] {meta['title']}")

rewritten = rewrite_query(vague_question)
print(f"\nRewritten query: {rewritten}")

print("\n--- Retrieval with rewritten question ---")
rewritten_results = retrieve(rewritten)
for meta in rewritten_results["metadatas"][0]:
    print(f"[{meta['category']}] {meta['title']}")
## hallucination test: model correctly said "not enough information" instead of guessing,
## but the Sources list still printed 3 unrelated articles -- retrieve() always returns its
## top-k nearest chunks regardless of relevance, so "sources" here isn't the same as "used"
##
## query rewriting: 2 of 3 retrieved articles were identical before and after rewriting,
## the third changed. no clear improvement or regression, hard to tell with only 30 articles
## in the collection, not enough variety for a genuinely bad vague-query result to compare against


--- RAG answer (off-topic question) ---
Not enough information.

Sources: ['Why Salesforce Stock Rallied Today', 'Delta Air Lines Is Up 10% This Year and Still Trades at Less Than 12 Times Earnings', 'Elon Musk Admits There Is Likely to Be Short-Term Pain for SpaceX Investors']

--- Retrieval with original vague question ---
[Technology] Before You Buy an AI Stock, Consider The Battle of The Cloud Titans
[Technology] Why The Direxion Daily Semiconductor Bull 3X ETF Plunged Nearly 20% Today
[Markets] Here Are the First 3 Stocks I'm Buying if the Market Crashes



Rewritten query: "Current trends and outlook for AI-focused publicly traded companies."

--- Retrieval with rewritten question ---
[Technology] Before You Buy an AI Stock, Consider The Battle of The Cloud Titans
[Markets] Here Are the First 3 Stocks I'm Buying if the Market Crashes
[Politics] What to Know About Nasdaq President Nelson Griggs Selling 3,226 Shares for $310,212


Off-topic question got "not enough information" instead of a made up answer. Sources still printed three unrelated articles though, retrieve() always returns its top-k nearest chunks no matter what, it doesn't know irrelevant as a concept, so sources here isn't the same as used.

Query rewriting swapped one of three retrieved articles after rephrasing the question, two stayed the same.

Hybrid search mixes semantic search (embeddings, what we've been doing) with keyword search. Semantic search can miss on rare terms, names, or exact numbers that don't embed distinctly, keyword search catches those instead.

Reranking happens after the first retrieval step. The embedding search grabs a batch of candidates, then a slower more precise model reorders just that smaller batch to put the best ones first.

Tool calling lets an LLM call an actual function instead of just generating text, a calculator, a database, an API, then read the result and keep going.

AI agents run that over multiple steps, retrieving stuff, calling tools, deciding what to do next based on what came back, instead of one prompt in and one answer out.

## takeaway

Built a retriever, a prompt template that pulls in chunks as context, source citations, a hallucination check, and query rewriting, all running off the ChromaDB collection from Day 4.